# Cross-Validation & Model Comparison

In previous notebooks, we compared models using a single train/test split.

That approach is useful, but the result depends on that particular split.

In this notebook, we'll use cross-validation to compare several classification models more reliably.

# Learning Objectives

By the end of this notebook, you will be able to:

- Explain k-fold cross-validation.
- Understand why cross-validation is useful.
- Use `cross_val_score`.
- Compare models using cross-validation.
- Interpret mean and standard deviation of CV scores.
- Understand why the test set should remain separate.

In [1]:
from sklearn.datasets import load_iris
import pandas as pd

data = load_iris(as_frame=True)

df = data.frame

df.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


In [2]:
X = df.drop(columns='target')
y = df['target']   

# Cross-Validation & Model Comparison

In previous notebooks, we compared models using a single train/test split.

That approach is useful, but the result depends on that particular split.

In this notebook, we'll use cross-validation to compare several classification models more reliably.

In [3]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    random_state=42,
)

scores = cross_val_score(
    model,
    X,
    y,
    cv=5,
)

scores

array([0.96666667, 1.        , 0.93333333, 0.96666667, 1.        ])

In [4]:
print(f"Mean accuracy: {scores.mean():.4f}")
print(f"Standard deviation: {scores.std():.4f}")

Mean accuracy: 0.9733
Standard deviation: 0.0249


The individual scores show performance on each fold.

The mean gives the average cross-validation accuracy.

The standard deviation shows how much the scores vary between folds.

In [5]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=42)),
    ]),
    
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=5)),
    ]),
    
    "Naive Bayes": GaussianNB(),
    
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
    ),
    
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC()),
    ]),
}

In [6]:
results = []

for name, model in models.items():
    scores = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring="accuracy",
    )
    
    results.append(
        {
            "Model": name,
            "Mean Accuracy": scores.mean(),
            "Std": scores.std(),
        }
    )

comparison = pd.DataFrame(results)

comparison.sort_values(
    "Mean Accuracy",
    ascending=False,
)

,Model,Mean Accuracy,Std
4,Random Forest,0.966667,0.021082
5,SVM,0.966667,0.021082
0,Logistic Regression,0.960000,0.038873
1,KNN,0.960000,0.024944
3,Decision Tree,0.953333,0.033993
2,Naive Bayes,0.953333,0.026667


# Interpreting Model Comparison

We can now compare models using the same cross-validation procedure.

The mean accuracy provides an estimate of average performance across the folds.

The standard deviation shows how much performance varied between folds.

This is a better basis for model comparison than simply comparing one test-set score.

# Interpreting the Model Comparison

The 5-fold cross-validation results were:

| Model | Mean Accuracy | Standard Deviation |
|---|---:|---:|
| Random Forest | 0.9667 | 0.0211 |
| SVM | 0.9667 | 0.0211 |
| Logistic Regression | 0.9600 | 0.0389 |
| KNN | 0.9600 | 0.0249 |
| Decision Tree | 0.9533 | 0.0340 |
| Naive Bayes | 0.9533 | 0.0267 |

Random Forest and SVM achieved the highest mean accuracy in this experiment.

However, their performance is tied, so we cannot claim that one is better than the other from these results.

The differences between the models are also relatively small.

These results apply only to this dataset and this cross-validation experiment. They should not be interpreted as a universal ranking of classification algorithms.

# Cross-Validation With a Held-Out Test Set

In a real Machine Learning workflow, we normally keep a final test set untouched.

Cross-validation is performed on the training data to select or compare models.

The final test set is used only after the model-selection process is complete.

This prevents the test set from influencing model selection.

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

In [8]:
cv_results = []

for name, model in models.items():
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="accuracy",
    )

    cv_results.append(
        {
            "Model": name,
            "Mean CV Accuracy": scores.mean(),
            "Std": scores.std(),
        }
    )

training_comparison = pd.DataFrame(cv_results)

training_comparison.sort_values(
    "Mean CV Accuracy",
    ascending=False,
)

,Model,Mean CV Accuracy,Std
1,KNN,0.966667,0.031180
5,SVM,0.966667,0.031180
0,Logistic Regression,0.958333,0.026352
2,Naive Bayes,0.958333,0.026352
4,Random Forest,0.950000,0.016667
3,Decision Tree,0.941667,0.020412
